In [1]:

# ============================================================
# TASK 2: RAG FROM SCRATCH
# Document: NIST AI Risk Management Framework 1.0
# Dense retrieval: Sentence Transformers + NumPy
# Comparison: BM25 + Cross-Encoder Reranking
# ============================================================

# 1. INSTALL LIBRARIES
!pip install -q pypdf sentence-transformers rank-bm25 pandas numpy

import os
import re
import json
import shutil
import numpy as np
import pandas as pd

from google.colab import files, drive
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from sklearn.preprocessing import normalize

# 2. MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Tndrr_Task2_RAG"
os.makedirs(PROJECT_DIR, exist_ok=True)

# 3. UPLOAD THE PUBLIC PDF
print("Upload NIST.AI.100-1.pdf")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No PDF uploaded.")

PDF_PATH = list(uploaded.keys())[0]

if not PDF_PATH.lower().endswith(".pdf"):
    raise ValueError("Please upload a PDF document.")

# Save a copy to Drive
DRIVE_PDF_PATH = os.path.join(PROJECT_DIR, "source_document.pdf")
shutil.copy(PDF_PATH, DRIVE_PDF_PATH)

# 4. EXTRACT PDF TEXT PAGE BY PAGE
reader = PdfReader(PDF_PATH)

print("\nPDF pages:", len(reader.pages))

if len(reader.pages) < 30:
    raise ValueError("The source document must contain at least 30 pages.")

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    text = re.sub(r'\s+', ' ', text).strip()

    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

print("Pages with extractable text:", len(pages))

if len(pages) < 30:
    raise ValueError(
        "Fewer than 30 pages contain extractable text. "
        "Check whether the PDF is scanned."
    )

with open(os.path.join(PROJECT_DIR, "extracted_pages.json"), "w") as f:
    json.dump(pages, f, indent=2)

# 5. DEFINE 15 EVALUATION QUESTIONS
# Anchor phrases are used to identify candidate evidence pages.
# Inspect the printed page numbers and passages before reporting
# the evaluation as manually validated ground truth.

questions = [
    {
        "id": 1,
        "question": "What is the purpose of the NIST AI Risk Management Framework?",
        "anchors": ["AI RMF", "risk management"]
    },
    {
        "id": 2,
        "question": "What are the four core functions of the AI RMF?",
        "anchors": ["GOVERN", "MAP", "MEASURE", "MANAGE"]
    },
    {
        "id": 3,
        "question": "What is the purpose of the GOVERN function?",
        "anchors": ["GOVERN"]
    },
    {
        "id": 4,
        "question": "What activities are included in the MAP function?",
        "anchors": ["MAP"]
    },
    {
        "id": 5,
        "question": "What is the purpose of the MEASURE function?",
        "anchors": ["MEASURE"]
    },
    {
        "id": 6,
        "question": "What is the purpose of the MANAGE function?",
        "anchors": ["MANAGE"]
    },
    {
        "id": 7,
        "question": "Why should AI systems be valid and reliable?",
        "anchors": ["valid and reliable"]
    },
    {
        "id": 8,
        "question": "What does safety mean in the context of trustworthy AI?",
        "anchors": ["safe", "safety"]
    },
    {
        "id": 9,
        "question": "How are security and resilience considered in trustworthy AI?",
        "anchors": ["secure and resilient"]
    },
    {
        "id": 10,
        "question": "Why are accountability and transparency important in AI?",
        "anchors": ["accountable", "transparent"]
    },
    {
        "id": 11,
        "question": "What is the difference between explainability and interpretability?",
        "anchors": ["explainable", "interpretable"]
    },
    {
        "id": 12,
        "question": "How does the framework address privacy?",
        "anchors": ["privacy"]
    },
    {
        "id": 13,
        "question": "How should fairness and harmful bias be considered?",
        "anchors": ["fair", "bias"]
    },
    {
        "id": 14,
        "question": "Who are AI actors in the AI risk management process?",
        "anchors": ["AI actors"]
    },
    {
        "id": 15,
        "question": "How should organizations determine their risk tolerance?",
        "anchors": ["risk tolerance"]
    }
]

# 6. CREATE PAGE-AWARE CHUNKS
def create_chunks(pages, chunk_size, overlap=50):

    chunks = []

    for page in pages:

        words = page["text"].split()

        start = 0

        while start < len(words):

            end = min(start + chunk_size, len(words))

            chunk_text = " ".join(words[start:end])

            chunks.append({
                "chunk_id": len(chunks),
                "page": page["page"],
                "text": chunk_text,
                "word_count": len(chunk_text.split())
            })

            if end == len(words):
                break

            start = end - overlap

    return chunks


chunks_300 = create_chunks(pages, 300, 50)
chunks_600 = create_chunks(pages, 600, 50)

print("\n300-word chunks:", len(chunks_300))
print("600-word chunks:", len(chunks_600))

# 7. CREATE CANDIDATE GOLD EVIDENCE PAGES
# A page is a candidate if it contains at least one anchor phrase.
# These are weak labels, not independently verified relevance labels.
# Inspect the candidates and correct them if necessary.

def find_candidate_pages(question, pages):

    matched_pages = []

    for page in pages:

        page_text = page["text"].lower()

        if any(
            anchor.lower() in page_text
            for anchor in question["anchors"]
        ):
            matched_pages.append(page["page"])

    return matched_pages


for q in questions:
    q["gold_pages"] = find_candidate_pages(q, pages)

print("\n===== CANDIDATE EVIDENCE PAGES =====")

for q in questions:
    print(
        f"Q{q['id']}: {q['question']}\n"
        f"Candidate pages: {q['gold_pages']}\n"
    )

# Stop if any question has no candidate page.
missing = [q["id"] for q in questions if not q["gold_pages"]]

if missing:
    raise ValueError(
        f"No candidate evidence page for questions {missing}. "
        "Inspect the PDF text and adjust their anchor phrases."
    )

# 8. LOAD EMBEDDING MODEL
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("\nEmbedding model loaded:", EMBEDDING_MODEL_NAME)

# 9. CREATE EMBEDDINGS AND NUMPY COSINE RETRIEVAL
def create_embeddings(chunks):

    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    embeddings = normalize(embeddings)

    return embeddings


def dense_search(question, chunks, embeddings, top_k=5):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    query_embedding = normalize(query_embedding)

    scores = np.dot(
        embeddings,
        query_embedding.T
    ).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:

        results.append({
            **chunks[index],
            "score": float(scores[index])
        })

    return results


# 10. RECALL@5 EVALUATION
# Success means at least one of the top-5 retrieved chunks
# comes from a candidate evidence page for that question.

def evaluate_recall(chunks, embeddings, questions, top_k=5):

    results = []

    for q in questions:

        retrieved = dense_search(
            q["question"],
            chunks,
            embeddings,
            top_k
        )

        gold_pages = set(q["gold_pages"])

        hit = any(
            item["page"] in gold_pages
            for item in retrieved
        )

        results.append({
            "question_id": q["id"],
            "question": q["question"],
            "gold_pages": sorted(gold_pages),
            "retrieved_pages": [
                item["page"] for item in retrieved
            ],
            "recall_hit": int(hit),
            "retrieved": retrieved
        })

    recall = (
        sum(x["recall_hit"] for x in results)
        / len(results)
    ) * 100

    return recall, results


# 11. EVALUATE 300-WORD CHUNKS
print("\nCreating 300-word embeddings...")
embeddings_300 = create_embeddings(chunks_300)

recall_300, results_300 = evaluate_recall(
    chunks_300,
    embeddings_300,
    questions
)

print("\n300-WORD CHUNK RECALL@5:", round(recall_300, 2), "%")

# 12. EVALUATE 600-WORD CHUNKS
print("\nCreating 600-word embeddings...")
embeddings_600 = create_embeddings(chunks_600)

recall_600, results_600 = evaluate_recall(
    chunks_600,
    embeddings_600,
    questions
)

print("\n600-WORD CHUNK RECALL@5:", round(recall_600, 2), "%")

# 13. PRINT RETRIEVED SOURCE PASSAGES
def display_results(results, label):

    print("\n======================================")
    print(label)
    print("======================================")

    for result in results:

        print("\nQuestion:", result["question"])
        print("Candidate evidence pages:", result["gold_pages"])
        print("Retrieved pages:", result["retrieved_pages"])
        print("Recall hit:", result["recall_hit"])

        for rank, item in enumerate(result["retrieved"][:3], 1):

            print(f"\nRank {rank} | Page {item['page']} | "
                  f"Score {item['score']:.4f}")

            print(item["text"][:900])


display_results(results_300, "300-WORD CHUNKS")
display_results(results_600, "600-WORD CHUNKS")

# 14. BM25 RETRIEVAL BASELINE
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


def bm25_evaluate(chunks, questions, top_k=5):

    corpus = [
        tokenize(chunk["text"])
        for chunk in chunks
    ]

    bm25 = BM25Okapi(corpus)

    results = []

    for q in questions:

        scores = bm25.get_scores(
            tokenize(q["question"])
        )

        top_indices = np.argsort(scores)[::-1][:top_k]

        retrieved = [
            chunks[index] for index in top_indices
        ]

        hit = any(
            item["page"] in set(q["gold_pages"])
            for item in retrieved
        )

        results.append({
            "question_id": q["id"],
            "question": q["question"],
            "retrieved_pages": [
                item["page"] for item in retrieved
            ],
            "recall_hit": int(hit)
        })

    recall = (
        sum(x["recall_hit"] for x in results)
        / len(results)
    ) * 100

    return recall, results


bm25_recall_300, bm25_results_300 = bm25_evaluate(
    chunks_300,
    questions
)

bm25_recall_600, bm25_results_600 = bm25_evaluate(
    chunks_600,
    questions
)

print("\n===== BM25 RESULTS =====")
print("300-word BM25 Recall@5:", round(bm25_recall_300, 2), "%")
print("600-word BM25 Recall@5:", round(bm25_recall_600, 2), "%")

# 15. OPTIONAL CROSS-ENCODER RERANKER
# Rerank the top 20 dense candidates and evaluate the best 5.

print("\nLoading cross-encoder reranker...")

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def rerank_evaluate(chunks, embeddings, questions,
                    candidate_k=20, top_k=5):

    results = []

    for q in questions:

        candidates = dense_search(
            q["question"],
            chunks,
            embeddings,
            candidate_k
        )

        pairs = [
            [q["question"], item["text"]]
            for item in candidates
        ]

        scores = reranker.predict(pairs)

        for item, score in zip(candidates, scores):
            item["rerank_score"] = float(score)

        reranked = sorted(
            candidates,
            key=lambda x: x["rerank_score"],
            reverse=True
        )[:top_k]

        hit = any(
            item["page"] in set(q["gold_pages"])
            for item in reranked
        )

        results.append({
            "question_id": q["id"],
            "question": q["question"],
            "retrieved_pages": [
                item["page"] for item in reranked
            ],
            "recall_hit": int(hit),
            "retrieved": reranked
        })

    recall = (
        sum(x["recall_hit"] for x in results)
        / len(results)
    ) * 100

    return recall, results


rerank_recall_300, rerank_results_300 = rerank_evaluate(
    chunks_300,
    embeddings_300,
    questions
)

rerank_recall_600, rerank_results_600 = rerank_evaluate(
    chunks_600,
    embeddings_600,
    questions
)

print("\n===== RERANKER RESULTS =====")
print("300-word reranked Recall@5:",
      round(rerank_recall_300, 2), "%")
print("600-word reranked Recall@5:",
      round(rerank_recall_600, 2), "%")

# 16. COMPARISON TABLE
comparison = pd.DataFrame([
    {
        "Method": "Dense retrieval",
        "Chunk size": 300,
        "Recall@5 (%)": round(recall_300, 2)
    },
    {
        "Method": "Dense retrieval",
        "Chunk size": 600,
        "Recall@5 (%)": round(recall_600, 2)
    },
    {
        "Method": "BM25",
        "Chunk size": 300,
        "Recall@5 (%)": round(bm25_recall_300, 2)
    },
    {
        "Method": "BM25",
        "Chunk size": 600,
        "Recall@5 (%)": round(bm25_recall_600, 2)
    },
    {
        "Method": "Dense + reranker",
        "Chunk size": 300,
        "Recall@5 (%)": round(rerank_recall_300, 2)
    },
    {
        "Method": "Dense + reranker",
        "Chunk size": 600,
        "Recall@5 (%)": round(rerank_recall_600, 2)
    }
])

print("\n===== RETRIEVAL COMPARISON =====")
print(comparison.to_string(index=False))

comparison.to_csv(
    os.path.join(PROJECT_DIR, "retrieval_comparison.csv"),
    index=False
)

# 17. IDENTIFY RETRIEVAL FAILURES
failures_300 = [
    result for result in results_300
    if result["recall_hit"] == 0
]

failures_600 = [
    result for result in results_600
    if result["recall_hit"] == 0
]

print("\n===== 300-WORD RETRIEVAL FAILURES =====")
for failure in failures_300:
    print(
        "Q", failure["question_id"],
        "|", failure["question"],
        "| Candidate pages:", failure["gold_pages"],
        "| Retrieved:", failure["retrieved_pages"]
    )

print("\n===== 600-WORD RETRIEVAL FAILURES =====")
for failure in failures_600:
    print(
        "Q", failure["question_id"],
        "|", failure["question"],
        "| Candidate pages:", failure["gold_pages"],
        "| Retrieved:", failure["retrieved_pages"]
    )

# 18. SAVE ALL EXPERIMENT RESULTS
def serializable_results(results):

    output = []

    for result in results:

        row = {
            key: value
            for key, value in result.items()
            if key != "retrieved"
        }

        if "retrieved" in result:
            row["retrieved"] = result["retrieved"]

        output.append(row)

    return output


all_results = {
    "document": "NIST AI Risk Management Framework 1.0",
    "document_pages": len(reader.pages),
    "pages_with_text": len(pages),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "questions": questions,
    "dense_300": {
        "recall_at_5": recall_300,
        "results": serializable_results(results_300)
    },
    "dense_600": {
        "recall_at_5": recall_600,
        "results": serializable_results(results_600)
    },
    "bm25_300_recall_at_5": bm25_recall_300,
    "bm25_600_recall_at_5": bm25_recall_600,
    "reranker_300_recall_at_5": rerank_recall_300,
    "reranker_600_recall_at_5": rerank_recall_600,
    "comparison": comparison.to_dict(orient="records"),
    "failures_300": failures_300,
    "failures_600": failures_600
}

with open(
    os.path.join(PROJECT_DIR, "rag_experiment_results.json"),
    "w"
) as f:
    json.dump(all_results, f, indent=2)

with open(
    os.path.join(PROJECT_DIR, "evaluation_questions.json"),
    "w"
) as f:
    json.dump(questions, f, indent=2)

# Save chunk metadata and passages
for filename, chunks in [
    ("chunks_300.json", chunks_300),
    ("chunks_600.json", chunks_600)
]:
    with open(os.path.join(PROJECT_DIR, filename), "w") as f:
        json.dump(chunks, f, indent=2)

# 19. FINAL FILE CHECK
print("\n===== TASK 2 FILE CHECK =====")

for filename in os.listdir(PROJECT_DIR):
    path = os.path.join(PROJECT_DIR, filename)

    if os.path.isfile(path):
        print(
            filename,
            "|",
            round(os.path.getsize(path) / 1024, 2),
            "KB"
        )

print("\n===== TASK 2 COMPLETE =====")


Mounted at /content/drive
Upload NIST.AI.100-1.pdf


Saving NIST.AI.100-1.pdf to NIST.AI.100-1.pdf

PDF pages: 48
Pages with extractable text: 48

300-word chunks: 82
600-word chunks: 48

===== CANDIDATE EVIDENCE PAGES =====
Q1: What is the purpose of the NIST AI Risk Management Framework?
Candidate pages: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]

Q2: What are the four core functions of the AI RMF?
Candidate pages: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]

Q3: What is the purpose of the GOVERN function?
Candidate pages: [4, 5, 8, 10, 12, 21, 24, 25, 26, 27, 28, 29, 30, 32, 35, 36, 39, 40, 41, 42, 46, 47]

Q4: What activities are included in the MAP function?
Candidate pages: [4, 5, 8, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 43, 44, 46]

Q5: What is 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (19 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


===== RERANKER RESULTS =====
300-word reranked Recall@5: 100.0 %
600-word reranked Recall@5: 100.0 %

===== RETRIEVAL COMPARISON =====
          Method  Chunk size  Recall@5 (%)
 Dense retrieval         300         100.0
 Dense retrieval         600         100.0
            BM25         300         100.0
            BM25         600         100.0
Dense + reranker         300         100.0
Dense + reranker         600         100.0

===== 300-WORD RETRIEVAL FAILURES =====

===== 600-WORD RETRIEVAL FAILURES =====

===== TASK 2 FILE CHECK =====
source_document.pdf | 1900.51 KB
extracted_pages.json | 106.87 KB
retrieval_comparison.csv | 0.16 KB
rag_experiment_results.json | 330.07 KB
evaluation_questions.json | 5.37 KB
chunks_300.json | 122.99 KB
chunks_600.json | 108.87 KB

===== TASK 2 COMPLETE =====


In [7]:

from sentence_transformers import SentenceTransformer
import numpy as np

# Reload the same embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")
print("300-word chunks:", len(chunks_300))
print("300-word embeddings:", embeddings_300.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.
300-word chunks: 82
300-word embeddings: (82, 384)


In [8]:

# TASK 2: ADDITIONAL RETRIEVAL STRESS TEST
# Does not overwrite the original benchmark.

stress_questions = [
    "What specific practices help organizations monitor AI systems after deployment and respond to incidents?",
    "What are the three major categories of bias identified by NIST, and how do they differ?",
    "What does the framework recommend when an AI system operates beyond its knowledge limits?"
]

for question in stress_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)

    # Uses the existing 300-word chunks and embeddings
    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = np.dot(embeddings_300, query_embedding[0])
    top_indices = np.argsort(scores)[::-1][:5]

    for rank, idx in enumerate(top_indices, start=1):
        chunk = chunks_300[idx]
        print(f"\nRank {rank} | Page {chunk['page']} | Score {scores[idx]:.4f}")
        print(chunk["text"][:900])



QUESTION: What specific practices help organizations monitor AI systems after deployment and respond to incidents?

Rank 1 | Page 29 | Score 0.6083
NIST AI 100-1 AI RMF 1.0 Table 1: Categories and subcategories for the GOVERN function. (Continued) that considers and communicates AI risk. GOVERN 4.2: Organizational teams document the risks and po- tential impacts of the AI technology they design, develop, deploy, evaluate, and use, and they communicate about the impacts more broadly. GOVERN 4.3: Organizational practices are in place to enable AI testing, identification of incidents, and information sharing. GOVERN 5: Processes are in place for robust engagement with relevant AI actors. GOVERN 5.1: Organizational policies and practices are in place to collect, consider, prioritize, and integrate feedback from those external to the team that developed or deployed the AI system regarding the potential individual and societal impacts related to AI risks. GOVERN 5.2: Mechanisms are establis

In [10]:
# Query expansion experiment for Question 3

original_question = (
    "What does the framework recommend when an AI system operates beyond its knowledge limits?"
)

expanded_question = (
    "AI system beyond knowledge limits, fail safely, "
    "safety risks, residual negative risk, system reliability, "
    "robustness, safe operation and failure response."
)

def retrieve_top5(question):
    q_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = np.dot(embeddings_300, q_embedding)
    indices = np.argsort(scores)[::-1][:5]

    for rank, idx in enumerate(indices, 1):
        chunk = chunks_300[idx]
        print(
            f"Rank {rank} | Page {chunk['page']} | "
            f"Score {scores[idx]:.4f}"
        )
        print(chunk["text"][:700])
        print()

print("===== ORIGINAL QUERY =====")
retrieve_top5(original_question)

print("===== EXPANDED QUERY =====")
retrieve_top5(expanded_question)

===== ORIGINAL QUERY =====
Rank 1 | Page 44 | Score 0.6019
For example, existing frameworks and guidance are unable to: • adequately manage the problem of harmful bias in AI systems; • confront the challenging risks related to generative AI; • comprehensively address security concerns related to evasion, model extraction, mem- bership inference, availability, or other machine learning attacks; • account for the complex attack surface of AI systems or other security abuses enabled by AI systems; and • consider risks associated with third-party AI technologies, transfer learning, and off- label use where AI systems may be trained for decision-making outside an organiza- tion’s security controls or trained in one domain and then “fine-tuned” for anothe

Rank 2 | Page 6 | Score 0.5665
NIST AI 100-1 AI RMF 1.0 Executive Summary Artificial intelligence (AI) technologies have significant potential to transform society and people’s lives – from commerce and health to transportation and cyberse

In [11]:
import json

failure_analysis = {
    "question": (
        "What does the framework recommend when an AI system "
        "operates beyond its knowledge limits?"
    ),
    "document": "NIST AI RMF 1.0",
    "initial_retrieval": {
        "relevant_page": 35,
        "retrieved_in_top5": False,
        "failure_reason": (
            "The original semantic query did not retrieve the "
            "passage discussing safe failure beyond knowledge limits."
        )
    },
    "improvement": "Query expansion",
    "expanded_query": (
        "AI system beyond knowledge limits, fail safely, "
        "safety risks, residual negative risk, system reliability, "
        "robustness, safe operation and failure response."
    ),
    "improved_retrieval": {
        "relevant_page": 35,
        "retrieved_in_top5": True,
        "rank": 3,
        "similarity_score": 0.7101
    },
    "conclusion": (
        "Adding domain-specific terms improved retrieval by "
        "bringing the relevant passage into the top five."
    )
}

with open("task2_failure_analysis.json", "w") as f:
    json.dump(failure_analysis, f, indent=4)

print("Failure analysis saved successfully.")

Failure analysis saved successfully.


In [9]:

# Retrieval improvement experiment: query expansion

original_question = (
    "What does the framework recommend when an AI system operates beyond its knowledge limits?"
)

expanded_question = (
    "AI system beyond knowledge limits, fail safely, "
    "safety risks, residual negative risk, system reliability, "
    "robustness, safe operation and failure response."
)

def retrieve_top5(question):
    q_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = np.dot(embeddings_300, q_embedding)
    indices = np.argsort(scores)[::-1][:5]

    for rank, idx in enumerate(indices, 1):
        chunk = chunks_300[idx]
        print(
            f"Rank {rank} | Page {chunk['page']} | "
            f"Score {scores[idx]:.4f}"
        )
        print(chunk["text"][:700])
        print()

print("===== ORIGINAL QUERY =====")
retrieve_top5(original_question)

print("===== EXPANDED QUERY =====")
retrieve_top5(expanded_question)


===== ORIGINAL QUERY =====
Rank 1 | Page 44 | Score 0.6019
For example, existing frameworks and guidance are unable to: • adequately manage the problem of harmful bias in AI systems; • confront the challenging risks related to generative AI; • comprehensively address security concerns related to evasion, model extraction, mem- bership inference, availability, or other machine learning attacks; • account for the complex attack surface of AI systems or other security abuses enabled by AI systems; and • consider risks associated with third-party AI technologies, transfer learning, and off- label use where AI systems may be trained for decision-making outside an organiza- tion’s security controls or trained in one domain and then “fine-tuned” for anothe

Rank 2 | Page 6 | Score 0.5665
NIST AI 100-1 AI RMF 1.0 Executive Summary Artificial intelligence (AI) technologies have significant potential to transform society and people’s lives – from commerce and health to transportation and cyberse

In [12]:
question_2 = (
    "How should organizations assess whether AI systems "
    "are fair across different demographic groups?"
)

q_embedding = model.encode(
    [question_2],
    convert_to_numpy=True,
    normalize_embeddings=True
)[0]

scores = np.dot(embeddings_300, q_embedding)
indices = np.argsort(scores)[::-1][:5]

print("QUESTION:", question_2)

for rank, idx in enumerate(indices, 1):
    chunk = chunks_300[idx]
    print(f"\nRank {rank} | Page {chunk['page']} | Score {scores[idx]:.4f}")
    print(chunk["text"][:700])

QUESTION: How should organizations assess whether AI systems are fair across different demographic groups?

Rank 1 | Page 22 | Score 0.6564
minimizing methods such as de-identification and aggregation for certain model outputs, can support design for privacy-enhanced AI systems. Under certain conditions such as data sparsity, privacy- enhancing techniques can result in a loss in accuracy, affecting decisions about fairness and other values in certain domains. 3.7 Fair – with Harmful Bias Managed Fairness in AI includes concerns for equality and equity by addressing issues such as harm- ful bias and discrimination. Standards of fairness can be complex and difficult to define be- cause perceptions of fairness differ among cultures and may shift depending on application. Organizations’ risk management efforts will be enhanced by re

Rank 2 | Page 42 | Score 0.5542
civil society organizations. The general public is most likely to directly experience positive and negative impacts of AI tech

In [13]:
expanded_question_2 = (
    "AI fairness evaluation demographic groups, "
    "measure disparities across populations, "
    "disaggregated performance metrics, "
    "representative datasets, equality, equity, "
    "harmful bias and discrimination."
)

print("===== ORIGINAL QUERY =====")
retrieve_top5(question_2)

print("===== EXPANDED QUERY =====")
retrieve_top5(expanded_question_2)

===== ORIGINAL QUERY =====
Rank 1 | Page 22 | Score 0.6564
minimizing methods such as de-identification and aggregation for certain model outputs, can support design for privacy-enhanced AI systems. Under certain conditions such as data sparsity, privacy- enhancing techniques can result in a loss in accuracy, affecting decisions about fairness and other values in certain domains. 3.7 Fair – with Harmful Bias Managed Fairness in AI includes concerns for equality and equity by addressing issues such as harm- ful bias and discrimination. Standards of fairness can be complex and difficult to define be- cause perceptions of fairness differ among cultures and may shift depending on application. Organizations’ risk management efforts will be enhanced by re

Rank 2 | Page 42 | Score 0.5542
civil society organizations. The general public is most likely to directly experience positive and negative impacts of AI technologies. They may provide the motivation for actions taken by the AI actors. Thi

In [14]:
targeted_question = (
    "disaggregated evaluation of AI system outcomes across demographic "
    "groups, compare false positive rates and false negative rates, "
    "measure disparities, fairness metrics, representative data, "
    "harmful bias assessment"
)

print("===== TARGETED FAIRNESS QUERY =====")
retrieve_top5(targeted_question)

===== TARGETED FAIRNESS QUERY =====
Rank 1 | Page 22 | Score 0.6710
minimizing methods such as de-identification and aggregation for certain model outputs, can support design for privacy-enhanced AI systems. Under certain conditions such as data sparsity, privacy- enhancing techniques can result in a loss in accuracy, affecting decisions about fairness and other values in certain domains. 3.7 Fair – with Harmful Bias Managed Fairness in AI includes concerns for equality and equity by addressing issues such as harm- ful bias and discrimination. Standards of fairness can be complex and difficult to define be- cause perceptions of fairness differ among cultures and may shift depending on application. Organizations’ risk management efforts will be enhanced by re

Rank 2 | Page 23 | Score 0.6274
NIST AI 100-1 AI RMF 1.0 Bias is broader than demographic balance and data representativeness. NIST has identified three major categories of AI bias to be considered and managed: systemic, computati

In [15]:
# Inspect the complete fairness-related chunks
for i, chunk in enumerate(chunks_300):
    if chunk["page"] in [22, 23]:
        print(f"\n===== CHUNK {i} | PAGE {chunk['page']} =====")
        print(chunk["text"])


===== CHUNK 33 | PAGE 22 =====
NIST AI 100-1 AI RMF 1.0 Risks to interpretability often can be addressed by communicating a description of why an AI system made a particular prediction or recommendation. (See “Four Principles of Explainable Artificial Intelligence” and “Psychological Foundations of Explainability and Interpretability in Artificial Intelligence” found here.) Transparency, explainability, and interpretability are distinct characteristics that support each other. Transparency can answer the question of “what happened” in the system. Ex- plainability can answer the question of “how” a decision was made in the system. Inter- pretability can answer the question of “why” a decision was made by the system and its meaning or context to the user. 3.6 Privacy-Enhanced Privacy refers generally to the norms and practices that help to safeguard human autonomy, identity, and dignity. These norms and practices typically address freedom from intrusion, limiting observation, or individ

In [16]:
question_3 = (
    "What methods should organizations use to evaluate AI system "
    "safety, including testing, monitoring and human intervention?"
)

print("===== SAFETY RETRIEVAL TEST =====")
retrieve_top5(question_3)

===== SAFETY RETRIEVAL TEST =====
Rank 1 | Page 19 | Score 0.6690
and should take into con- sideration that certain types of failures can cause greater harm. AI risk management efforts should prioritize the minimization of potential negative impacts, and may need to include human intervention in cases where the AI system cannot detect or correct errors. 3.2 Safe AI systems should “not under defined conditions, lead to a state in which human life, health, property, or the environment is endangered” (Source: ISO/IEC TS 5723:2022). Safe operation of AI systems is improved through: • responsible design, development, and deployment practices; • clear information to deployers on responsible use of the system; • responsible decision-making by deployers and end us

Rank 2 | Page 33 | Score 0.6511
methodolo- gies are in place, followed, and documented. Metrics and measurement methodologies should adhere to scientific, legal, and ethical norms and be carried out in an open and trans- parent proc

In [18]:
import os
import json

# Locate the saved evaluation questions file
matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "evaluation_questions.json" in files:
        matches.append(os.path.join(root, "evaluation_questions.json"))

if matches:
    questions_path = matches[0]
    with open(questions_path, "r", encoding="utf-8") as f:
        evaluation_questions = json.load(f)

    print("Loaded:", questions_path)
    print("Total questions:", len(evaluation_questions))
    print("\nFirst question:")
    print(evaluation_questions[0])
else:
    print("File not found in Google Drive.")

Loaded: /content/drive/MyDrive/Tndrr_Task2_RAG/evaluation_questions.json
Total questions: 15

First question:
{'id': 1, 'question': 'What is the purpose of the NIST AI Risk Management Framework?', 'anchors': ['AI RMF', 'risk management'], 'gold_pages': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]}


In [19]:
for item in evaluation_questions:
    print(
        f"Q{item['id']}: {item['question']}\n"
        f"Current gold pages: {item['gold_pages']}\n"
        f"Number of gold pages: {len(item['gold_pages'])}\n"
    )

Q1: What is the purpose of the NIST AI Risk Management Framework?
Current gold pages: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
Number of gold pages: 47

Q2: What are the four core functions of the AI RMF?
Current gold pages: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
Number of gold pages: 46

Q3: What is the purpose of the GOVERN function?
Current gold pages: [4, 5, 8, 10, 12, 21, 24, 25, 26, 27, 28, 29, 30, 32, 35, 36, 39, 40, 41, 42, 46, 47]
Number of gold pages: 22

Q4: What activities are included in the MAP function?
Current gold pages: [4, 5, 8, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 43, 44, 46]
Number of gold pages: 21

Q5: What is the purpose of the MEASURE function?
Current gold pages: [4,

In [20]:
# Manually curated primary evidence pages from NIST AI RMF

gold_pages = {
    1: [1, 2, 3, 6],
    2: [5, 6],
    3: [24, 25, 26, 27, 28, 29],
    4: [30, 31, 32],
    5: [33, 34, 35, 36],
    6: [37, 38, 39, 40],
    7: [17, 18],
    8: [19, 20],
    9: [20, 21],
    10: [20, 21, 27],
    11: [21, 22],
    12: [22],
    13: [22, 23],
    14: [7, 13, 14, 15, 16],
    15: [12, 27, 31, 34, 35, 38]
}

for item in evaluation_questions:
    item["gold_pages"] = gold_pages[item["id"]]

with open(questions_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_questions, f, indent=4)

print("Updated evaluation labels.")
print("Questions:", len(evaluation_questions))
print("Saved:", questions_path)

Updated evaluation labels.
Questions: 15
Saved: /content/drive/MyDrive/Tndrr_Task2_RAG/evaluation_questions.json


In [21]:
def calculate_recall_at_5(chunks, embeddings, questions):
    scores_list = []

    for item in questions:
        q_emb = model.encode(
            [item["question"]],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        scores = np.dot(embeddings, q_emb)
        top_indices = np.argsort(scores)[::-1][:5]

        retrieved_pages = {
            chunks[i]["page"] for i in top_indices
        }

        relevant_pages = set(item["gold_pages"])

        recall = (
            len(retrieved_pages & relevant_pages)
            / len(relevant_pages)
        )

        scores_list.append(recall)

    return round(np.mean(scores_list) * 100, 2)


recall_300 = calculate_recall_at_5(
    chunks_300, embeddings_300, evaluation_questions
)

recall_600 = calculate_recall_at_5(
    chunks_600, embeddings_600, evaluation_questions
)

print("300-word Recall@5:", recall_300, "%")
print("600-word Recall@5:", recall_600, "%")

300-word Recall@5: 65.44 %
600-word Recall@5: 49.33 %


In [22]:
import json
import os

output_dir = "/content/drive/MyDrive/Tndrr_Task2_RAG"

summary = {
    "document": "NIST AI RMF 1.0",
    "document_pages": 48,
    "questions_evaluated": 15,
    "chunk_sizes": [300, 600],
    "evaluation": {
        "method": "Dense retrieval Recall@5",
        "relevance_labels": "Manually selected primary evidence pages; provisional",
        "recall_at_5_300_words": recall_300,
        "recall_at_5_600_words": recall_600
    },
    "retrieval_methods": [
        "Dense retrieval",
        "BM25",
        "Dense retrieval with cross-encoder reranking"
    ],
    "failure_analysis": {
        "documented_failures": 1,
        "successful_fixes": [
            "Query expansion retrieved the relevant passage on page 35 at rank 3."
        ],
        "second_failure": "Not established"
    },
    "limitations": [
        "Relevance labels are manually selected and provisional.",
        "The reported Recall@5 comparison uses dense retrieval.",
        "Only one genuine retrieval failure and successful fix was verified."
    ]
}

with open(os.path.join(output_dir, "final_task2_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("TASK 2 SUMMARY SAVED")
print("Recall@5 (300 words):", recall_300)
print("Recall@5 (600 words):", recall_600)
print("Output directory:", output_dir)

TASK 2 SUMMARY SAVED
Recall@5 (300 words): 65.44
Recall@5 (600 words): 49.33
Output directory: /content/drive/MyDrive/Tndrr_Task2_RAG
